# CSoT'26 - ML in Astronomy - Week 2 . Part 2: Your First Neural Network (Solution)

Reference implementation. **Only open after attempting [`week2_mlp_starter.ipynb`](week2_mlp_starter.ipynb).**

Companion reading: [`04-building-models-with-nn-module.md`](../04-building-models-with-nn-module.md) and [`05-loss-functions-and-optimisers.md`](../05-loss-functions-and-optimisers.md).

## Step 0 — Re-create the Week 1 data pipeline

Week 2 builds directly on the `DataLoader`s from Week 1. The cells below reproduce that pipeline (download is commented out — uncomment it the first time, exactly as in [`week1_data_solution.ipynb`](../../Week-1/notebooks/week1_data_solution.ipynb)). If you saved `galaxy_data/` to Google Drive in Week 1, just re-mount Drive and point `DATA_ROOT` at it instead of re-downloading.

After this section you should have `train_loader`, `val_loader`, `test_loader`, `train_ds`, and `num_classes`.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

In [ ]:
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"   # adjust if your JPGs landed one folder deeper
DATA_ROOT = Path("galaxy_data")        # train/val/test subfolders get created here
LABELS_URL = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"

# --- Download via Kaggle API (run once; same as Week 1) ---
# from google.colab import files
# files.upload()  # select kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
# !unzip -q -o galaxy-zoo-2-images.zip -d {RAW_ROOT}
# !unzip -q -o {RAW_ROOT}/images_gz2.zip -d {IMAGES_DIR}
# !wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz {LABELS_URL}
# !gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz

print("RAW_ROOT   =", RAW_ROOT)
print("IMAGES_DIR =", IMAGES_DIR)
print("DATA_ROOT  =", DATA_ROOT)

In [ ]:
def high_level_label(gz2_class):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, ...) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def load_labeled_table(mapping_csv, labels_csv):
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    return df.dropna(subset=["label"]).reset_index(drop=True)


def _link_image(src, dst):
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(Path(src).resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(images_dir, df, out_root, per_class=200,
                                   train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    images_dir, out_root = Path(images_dir), Path(out_root)
    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)
        n = len(rows)
        n_train, n_val = int(train_frac * n), int(val_frac * n)
        splits = {
            "train": rows.iloc[:n_train],
            "val": rows.iloc[n_train:n_train + n_val],
            "test": rows.iloc[n_train + n_val:],
        }
        for split_name, split_rows in splits.items():
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists():
                    _link_image(src, dst)


PER_CLASS = 200  # bump up (e.g. 2000) once everything works

df = load_labeled_table(RAW_ROOT / "gz2_filename_mapping.csv", RAW_ROOT / "gz2_hart16.csv")
build_split_imagefolder_layout(IMAGES_DIR, df, DATA_ROOT, per_class=PER_CLASS)
print(df["label"].value_counts())

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(train_ds.classes)
print("classes      :", train_ds.classes)
print("class_to_idx :", train_ds.class_to_idx)
print("num_classes  :", num_classes)

## Step 1 - Define the model

A 2-layer MLP: flatten -> Linear(12288, 128) -> ReLU -> Linear(128, num_classes). The output layer returns **raw logits** (no softmax - `CrossEntropyLoss` adds it). Don't forget `super().__init__()`.

In [ ]:
class GalaxyMLP(nn.Module):
    def __init__(self, in_features=3 * 64 * 64, hidden=128, num_classes=3):
        super().__init__()                        # MUST be first
        self.flatten = nn.Flatten()               # (B,3,64,64) -> (B,12288)
        self.fc1 = nn.Linear(in_features, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, num_classes) # outputs raw logits

    def forward(self, x):                          # x: (B, 3, 64, 64)
        x = self.flatten(x)                        # (B, 12288)
        x = self.relu(self.fc1(x))                 # (B, 128)
        return self.fc2(x)                         # (B, num_classes)

## Step 2 - Instantiate and move to the device

Use the real `num_classes` from your data, and `.to(device)` so the model lives where the batches will.

In [ ]:
model = GalaxyMLP(num_classes=num_classes).to(device)
print("Model is on:", next(model.parameters()).device)

## Step 3 - Inspect the architecture

`print(model)` shows the layers; counting `.parameters()` shows how many weights there are. Notice that the first `Linear` (12288 x 128) dominates - a direct cost of flattening.

In [ ]:
print(model)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")

## Step 4 - Forward-pass one real batch

Pull a batch from `train_loader`, move it to the device, and run it through the model. The output should be logits of shape `(batch_size, num_classes)`.

In [ ]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

logits = model(images)
print("input  :", images.shape)
print("logits :", logits.shape)   # (32, num_classes)
assert logits.shape == (images.shape[0], num_classes)

## Step 5 - Loss and optimiser

`CrossEntropyLoss` consumes raw logits + integer labels. `Adam` with `lr=1e-3` is the sensible default.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print(criterion)
print(optimizer)

## Step 6 - Sanity-check the starting loss

For an untrained model on `C` balanced classes the loss should sit near `ln(C)`. If it's wildly off (or `nan`), suspect label dtype, a stray softmax, or unnormalised inputs.

In [ ]:
import math

loss = criterion(logits, labels)
print(f"loss            : {loss.item():.4f}")
print(f"ln(num_classes) : {math.log(num_classes):.4f}   <- expected ballpark")

## Step 7 - One optimisation step (learning, in miniature)

The three lines at the heart of every training loop: clear gradients, backprop, update. Re-evaluate the loss on the *same* batch afterwards - it should drop. (Week 3 repeats this over all batches for many epochs.)

In [ ]:
model.train()
loss_before = criterion(model(images), labels)

optimizer.zero_grad()   # 1. clear old gradients
loss_before.backward()  # 2. backpropagation
optimizer.step()        # 3. update weights

loss_after = criterion(model(images), labels)
print(f"loss before step : {loss_before.item():.4f}")
print(f"loss after  step : {loss_after.item():.4f}")
print("decreased!" if loss_after < loss_before else "did not decrease - check lr / setup")

## Step 8 (stretch) - How big does the model get?

Re-build the MLP with different hidden widths and print the parameter count each time. The first `Linear` (in_features x hidden) dominates - flattening is expensive. A CNN (Week 3) does more with far fewer parameters by sharing weights across the image.

In [ ]:
for hidden in (64, 128, 256, 512):
    m = GalaxyMLP(hidden=hidden, num_classes=num_classes)
    n = sum(p.numel() for p in m.parameters())
    print(f"hidden={hidden:4d}  ->  {n:,} parameters")

## Reflection (example answers)

1. **Starting loss.** An untrained model spreads probability roughly equally over `C` classes, so the true class gets ~`1/C` probability and the cross-entropy is `-ln(1/C) = ln(C)`.
2. **One step != trained.** We only lowered the loss on a *single* batch. Real training repeats `zero_grad -> backward -> step` over *all* batches for many epochs, and is judged on *held-out* data - that's Week 3.
3. **Why the first layer is huge.** Flattening connects every one of 12 288 inputs to every hidden unit, so `fc1` alone holds `12288 x hidden` weights. A CNN reuses small filters across the whole image, so it captures spatial structure with far fewer parameters.

---

That completes Week 2. Next: **Week 3 - CNNs, the Training Loop & Evaluation**, where this model becomes a CNN and actually gets trained.